In [1]:
import os
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("m4_01_read_mart_local")
    .master("local[*]")
    .getOrCreate()
)

from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important;}</style>"))

16:49:14 [WARN] o.a.h.u.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
16:49:15 [WARN] o.a.s.u.DependencyUtils - Local jar /jars/postgresql-42.7.4.jar does not exist, skipping.
16:49:15 [WARN] o.a.s.u.DependencyUtils - Local jar /jars/hadoop-aws-3.3.4.jar does not exist, skipping.
16:49:15 [WARN] o.a.s.u.DependencyUtils - Local jar /jars/aws-java-sdk-bundle-1.12.262.jar does not exist, skipping.
16:49:15 [WARN] o.a.s.SparkConf - Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
16:49:18 [ERROR] o.a.s.SparkContext - Failed to add file:/jars/postgresql-42.7.4.jar to Spark environment
java.io.FileNotFoundException: Jar /jars/postgresql-42.7.4.jar not found
	at org.apache.spark.SparkContext.addLocalJarFile$1(SparkContext.scala:2100) ~[spark-core_2.12-3.5.1.jar:3.5.1]
	at org.apache.spark.SparkContext.addJa

In [2]:
from pyspark.sql import functions as F

jdbc_url = "jdbc:postgresql://postgres:5432/dwh"
jdbc_props = {
    "user": "app",
    "password": "app",
    "driver": "org.postgresql.Driver",
}

f = spark.read.jdbc(jdbc_url, "core.fct_order_items", properties=jdbc_props)
p = spark.read.jdbc(jdbc_url, "core.dim_product", properties=jdbc_props)
s = spark.read.jdbc(jdbc_url, "core.dim_seller", properties=jdbc_props)
c = spark.read.jdbc(jdbc_url, "core.dim_customer", properties=jdbc_props)

In [3]:
from pyspark.sql import functions as F

f2 = f.withColumn("order_date", F.to_date(F.col("order_approved_at")))

wide = (
    f2.alias("f")
      .join(p.alias("p"), F.col("f.product_sk")  == F.col("p.product_sk"),  "left")
      .join(s.alias("s"), F.col("f.seller_sk")   == F.col("s.seller_sk"),   "left")
      .join(c.alias("c"), F.col("f.customer_sk") == F.col("c.customer_sk"), "left")
      .select(
          F.col("f.order_item_sk"),
          F.col("f.order_id"),
          F.col("f.order_item_id"),

          F.col("f.customer_sk").alias("customer_sk"),
          F.col("f.product_sk").alias("product_sk"),
          F.col("f.seller_sk").alias("seller_sk"),

          F.col("f.order_approved_at"),
          F.col("f.order_date"),
          F.col("f.shipping_limit_date"),
          F.col("f.price"),
          F.col("f.freight_value"),

          F.col("p.product_category_name").alias("product_category_name"),
          F.col("s.seller_city").alias("seller_city"),
          F.col("s.seller_state").alias("seller_state"),
          F.col("c.customer_city").alias("customer_city"),
          F.col("c.customer_state").alias("customer_state"),

          F.col("f.src_ingest_date"),
          F.current_timestamp().alias("load_dttm")
      )
)


In [4]:
wide.show(10,0)

+-------------+--------------------------------+-------------+-----------+----------+---------+-------------------+----------+-------------------+-------+-------------+---------------------------+-----------------+------------+--------------+--------------+---------------+--------------------------+
|order_item_sk|order_id                        |order_item_id|customer_sk|product_sk|seller_sk|order_approved_at  |order_date|shipping_limit_date|price  |freight_value|product_category_name      |seller_city      |seller_state|customer_city |customer_state|src_ingest_date|load_dttm                 |
+-------------+--------------------------------+-------------+-----------+----------+---------+-------------------+----------+-------------------+-------+-------------+---------------------------+-----------------+------------+--------------+--------------+---------------+--------------------------+
|1            |5f79b5b0931d63f1a42989eb65b9da6e|1            |41370      |13240     |269      |20